## 少样本示例的提示词模板使用

FewShotPromptTemplate:

FewShotChatMessagePromptTemplate:

Example selectors:

### 1.FewShotPromptTemplate:

In [ ]:
#例1：不使用示例
import os
import dotenv
from langchain_community.chat_models.tongyi import ChatTongyi

dotenv.load_dotenv()

os.environ['DASHSCOPE_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")


chat_model = ChatTongyi(model="qwen-flash",temperature=0.4)
res = chat_model.invoke("2 --\(˙<>˙)/-- 9%是多少?")
print(res.content)


你问的“2 --\(˙<>˙)/-- 9%是多少？”看起来像是一个带有表情符号或符号装饰的问题，但核心是“9%是多少”，我们来认真分析一下。

---

### 一、如果只是问：**9% 是多少？**

9% 表示 **9 除以 100**，也就是：

$$
9\% = \frac{9}{100} = 0.09
$$

✅ 所以，**9% 等于 0.09**（小数形式）。

---

### 二、如果你是在问：**2 的 9% 是多少？**

那就要计算：

$$
2 \times 9\% = 2 \times 0.09 = 0.18
$$

✅ 答案是：**0.18**

---

### 三、关于你的表达式 “2 --\(˙<>˙)/-- 9%”

这看起来像是用表情符号和符号玩文字游戏，比如：
- `--` 可能表示连接或运算符
- `\(˙<>˙)/--` 像是一个拟人化的表情（比如一个眨眼的猫或卡通脸）

但这在数学上没有标准意义。所以我们可以理解为：
> 你可能想用有趣的方式提问：“2 的 9% 是多少？”

---

### ✅ 最终答案（最合理的解释）：
> **2 的 9% 是 0.18**

---

如果你想表达的是别的意思（比如某种编码、谜题、梗），欢迎补充说明！😄


In [27]:
#例2：使用FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
#创建一个PromptTemplate实例
prompt_template = PromptTemplate.from_template(
    template="input: {input}\noutput: {output}",
)

#提供一些示例
examples = [ 
    {"input":"北京天气怎么样","output":"北京市"}, 
    {"input":"南京下雨吗","output":"南京市"}, 
    {"input":"武汉热吗","output":"武汉市"},
]


#创建一个FewShotPromptTemplate实例
few_shot_template = FewShotPromptTemplate(
    example_prompt=prompt_template,
    examples=examples,
    suffix="input: {input}\n output: ", #声明在示例后面的提示词模板
    input_variables=["input",],
    

)

#使用FewShotPromptTemplate实例生成提示词
few_shot_template.invoke({"input":"成都天气怎么样"})

StringPromptValue(text='input: 北京天气怎么样\noutput: 北京市\n\ninput: 南京下雨吗\noutput: 南京市\n\ninput: 武汉热吗\noutput: 武汉市\n\ninput: 成都天气怎么样\n output: ')

调用大模型

In [ ]:
#例3
#1、创建提示模板
from langchain_core.prompts import PromptTemplate
# 创建提示模板,配置一个提示模板,将一个示例格式化为字符串
prompt_template ="你是一个数学专家,算式:{input} 值:{output}使用:{description}"
# 这是一个提示模板,用于设置每个示例的格式
prompt_sample = PromptTemplate.from_template(prompt_template)

#2、提供示例
examples = [
    {"input": "2+2", "output": "4", "description": "加法运算"}, 
    {"input": "5-2", "output": "3", "description": "减法运算"},
]

#3、创建一个FewShotPromptTemplate对象
from langchain_core.prompts import FewShotPromptTemplate
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=prompt_sample,
    suffix="你是一个数学专家,算式:{input} 值:{output}",
    input_variables=["input", "output"]
)

import os
import dotenv
from langchain_community.chat_models.tongyi import ChatTongyi

dotenv.load_dotenv()




chat_model = ChatTongyi(
    model="qwen-flash",
    temperature=0.4)

result = chat_model.invoke(prompt.invoke({"input":"2*5", "output": "10"}))
print(result.content)


你是一个数学专家，算式：2 × 5  
值：10  
使用：乘法运算


### 2.FewShotChatMessagePromptTemplate的使用

In [ ]:
#实例化
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate
# 1. 示例消息格式
examples = [
    {"input":"1+1等于几?", "output":"1+1等于2"},
    {"input":"法国的首都是?", "output":"巴黎"}
]

# 2. 定义示例的消息格式提示词模版
msg_example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# 3. 创建 FewShotChatMessagePromptTemplate 对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=msg_example_prompt,
    examples=examples
)

# 4. 输出格式化后的消息
print(few_shot_prompt.format())
result1 = chat_model.invoke(few_shot_prompt.format())
print(result1.content)


Human: 1+1等于几?
AI: 1+1等于2
Human: 法国的首都是?
AI: 巴黎
1+1等于2。  
法国的首都是巴黎。


In [31]:
#
# 1. 导入相关包
from langchain_core.prompts import (FewShotChatMessagePromptTemplate, ChatPromptTemplate)

# 2. 定义示例组
examples = [
    {"input": "2 --\(˙<>˙)/-- 2", "output": "4"}, 
    {"input": "2 --\(˙<>˙)/-- 3", "output": "8"}
]

# 3. 定义示例的消息格式提示词模版
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input} 是多少?'),
    ('ai', '{output}')
])

# 4. 创建 FewShotChatMessagePromptTemplate 对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)
# 5. 输出完整提示词的消息模版
final_prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个数学奇才'),
    few_shot_prompt,
    ('human', '{input}')
])

# 6. 提供大模型
import os
import dotenv
from langchain_community.chat_models.tongyi import ChatTongyi

dotenv.load_dotenv()

chat_model = ChatTongyi(
    model="qwen-flash",
    temperature=0.4)

result = chat_model.invoke(final_prompt.invoke(input="2 --\(˙<>˙)/-- 4"))
print(result.content)

16


### 3. Example selectors

In [38]:
#例1
# 1. 导入相关包
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
import os
import dotenv
from langchain_community.embeddings.dashscope import DashScopeEmbeddings


dotenv.load_dotenv()

# 2. 定义嵌入模型
os.environ['DASHSCOPE_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")

embeddings_model = DashScopeEmbeddings(
    model="text-embedding-v4",
)

# 3. 定义示例组
examples = [
    {
        "question": "谁活得更久，穆罕默德·阿里还是艾伦·图灵？",
        "answer": """
接下来还需要问什么问题吗？
追问：穆罕默德·阿里去世时多大年纪？
中间答案：穆罕默德·阿里去世时享年74岁。
"""
    },
    {
        "question": "craigslist的创始人是什么时候出生的？",
        "answer": """
接下来还需要问什么问题吗？
追问：谁是craigslist的创始人？
中间答案：Craigslist是由克雷格·纽马克创立的。
"""
    },
    {
        "question": "谁是乔治·华盛顿的外祖父？",
        "answer": """
接下来还需要问什么问题吗？
追问：谁是乔治·华盛顿的母亲？
中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。
"""
    },
    {
        "question": "《大白鲨》和《皇家赌场》的导演都来自同一个国家吗？",
        "answer": """
接下来还需要问什么问题吗？
追问：《大白鲨》的导演是谁？
中间答案：《大白鲨》的导演是史蒂文·斯皮尔伯格。
"""
    }
]

# 4. 定义示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 这是可供选择的示例列表
    examples,
    # 这是用于生成嵌入的嵌入类，用于衡量语义相似性
    embeddings_model,
    # 这是用于存储嵌入并进行相似性搜索的 VectorStore 类
    Chroma,
    # 这是要生成的示例数量
    k=1,
)

# 选择与输入最相似的示例
question = "玛丽·鲍尔·华盛顿的父亲是谁？"
selected_examples = example_selector.select_examples({"question": question})
print(f"与输入最相似的示例: {selected_examples}")

# for example in selected_examples:
#     print("\n")
#     for k, v in example.items():
#         print(f"{k}: {v}")




与输入最相似的示例: [{'answer': '\n接下来还需要问什么问题吗？\n追问：谁是乔治·华盛顿的母亲？\n中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。\n', 'question': '谁是乔治·华盛顿的外祖父？'}]


In [43]:
#例2
# 1. 导入相关包
from langchain_community.vectorstores import FAISS
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_community.embeddings.dashscope import DashScopeEmbeddings


# 2. 定义示例提示词模版
example_prompt = PromptTemplate.from_template(
    template="Input: {input}\nOutput: {output}",
)

# 3. 创建一个示例提示词模版
examples = [
    {"input": "高兴", "output": "悲伤"},
    {"input": "高", "output": "矮"},
    {"input": "长", "output": "短"},
    {"input": "精力充沛", "output": "无精打采"},
    {"input": "阳光", "output": "阴暗"},
    {"input": "粗糙", "output": "光滑"},
    {"input": "干燥", "output": "潮湿"},
    {"input": "富裕", "output": "贫穷"},
]

# 4. 定义嵌入模型
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
)
# 5. 创建语义相似性示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    FAISS,
    k=2,
)

# 或者
# example_selector = SemanticSimilarityExampleSelector(
#     examples,
#     embeddings,
#     FAISS,
#     k=2
# )

# 6. 定义小样本提示词模版
similar_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个词组的反义词",
    suffix="Input: {word}\nOutput:",
    input_variables=["word"],
)

response = similar_prompt.format(word="快乐")
print(response)


给出每个词组的反义词

Input: 高兴
Output: 悲伤

Input: 阳光
Output: 阴暗

Input: 快乐
Output:
